# Camada BRONZE — Ingestão Bruta
Lê os CSVs e persiste como Delta Lake sem nenhuma transformação.

In [7]:
import sys
sys.path.insert(0, '/opt/spark/work-dir')
from tools.spark_session import get_spark

spark = get_spark('Bronze - Ingestão')
spark.sparkContext.setLogLevel('WARN')
print('Spark iniciado:', spark.version)

Spark iniciado: 4.0.0


26/06/21 02:42:12 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [8]:
# Gera os datasets de exemplo (roda só uma vez)
#import subprocess, sys
#subprocess.run([sys.executable, '/opt/spark/work-dir/datasets/gerar_dados.py'], check=True)

In [9]:
import glob as glob_mod

DATASETS = '/opt/spark/work-dir/datasets'
BRONZE   = '/opt/spark/work-dir/warehouse/bronze'

for table in ['clientes', 'produtos', 'vendas', 'fretes', 'pagamentos', 'cupons', 'avaliacoes', 'fornecedores']:
    files = sorted(glob_mod.glob(f'{DATASETS}/{table}_*.csv'))
    if not files:
        print(f'[bronze/{table}] NENHUM arquivo encontrado — pulando')
        continue
    df = (
        spark.read
        .option('header', True)
        .option('inferSchema', True)
        .csv(files[0])
    )
    df.write.format('delta').mode('overwrite').save(f'{BRONZE}/{table}')
    print(f'[bronze/{table}] {df.count()} linhas salvas  (fonte: {files[0]})')

[bronze/clientes] 100000 linhas salvas  (fonte: /opt/spark/work-dir/datasets/clientes_20260621T013728.csv)
[bronze/produtos] 1000 linhas salvas  (fonte: /opt/spark/work-dir/datasets/produtos_20260621T013728.csv)


26/06/21 02:42:18 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/06/21 02:42:18 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/06/21 02:42:18 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/06/21 02:42:18 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/06/21 02:42:18 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/06/21 02:42:18 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/06/21 02:42:18 WARN MemoryManager: Total allocation exceeds 95.

[bronze/vendas] 1000000 linhas salvas  (fonte: /opt/spark/work-dir/datasets/vendas_20260621T013728.csv)


26/06/21 02:42:22 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/06/21 02:42:22 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/06/21 02:42:22 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/06/21 02:42:22 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/06/21 02:42:22 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/06/21 02:42:23 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/06/21 02:42:23 WARN MemoryManager: Total allocation exceeds 95.0

[bronze/fretes] 1000000 linhas salvas  (fonte: /opt/spark/work-dir/datasets/fretes_20260621T013728.csv)


26/06/21 02:42:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/06/21 02:42:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/06/21 02:42:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/06/21 02:42:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/06/21 02:42:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


[bronze/pagamentos] 1000000 linhas salvas  (fonte: /opt/spark/work-dir/datasets/pagamentos_20260621T013728.csv)
[bronze/cupons] 200 linhas salvas  (fonte: /opt/spark/work-dir/datasets/cupons_20260621T013728.csv)


[bronze/avaliacoes] 420137 linhas salvas  (fonte: /opt/spark/work-dir/datasets/avaliacoes_20260621T013728.csv)
[bronze/fornecedores] 50 linhas salvas  (fonte: /opt/spark/work-dir/datasets/fornecedores_20260621T013728.csv)


In [10]:
# Inspeciona o schema de cada tabela
for table in ['clientes', 'produtos', 'vendas', 'fretes', 'pagamentos', 'cupons', 'avaliacoes', 'fornecedores']:
    df = spark.read.format('delta').load(f'{BRONZE}/{table}')
    print(f'\n=== {table.upper()} ===')
    df.printSchema()
    df.show(5, truncate=False)


=== CLIENTES ===
root
 |-- cliente_id: integer (nullable = true)
 |-- nome: string (nullable = true)
 |-- tipo_cliente: string (nullable = true)
 |-- documento: string (nullable = true)
 |-- regiao: string (nullable = true)
 |-- cidade: string (nullable = true)
 |-- data_cadastro: date (nullable = true)
 |-- ativo: integer (nullable = true)
 |-- score_credito: double (nullable = true)

+----------+------------------+------------+------------------+--------+-------------+-------------+-----+-------------+
|cliente_id|nome              |tipo_cliente|documento         |regiao  |cidade       |data_cadastro|ativo|score_credito|
+----------+------------------+------------+------------------+--------+-------------+-------------+-----+-------------+
|1         |João Pedro Sampaio|PF          |275.860.391-86    |Sul     |Porto Alegre |2023-04-19   |1    |788.9        |
|2         |Allana Moraes     |PF          |659.310.478-75    |Sul     |Caxias do Sul|2023-09-16   |1    |602.7        |
|3   

+------------+--------+----+-------------------------------+----------+
|avaliacao_id|venda_id|nota|comentario                     |data      |
+------------+--------+----+-------------------------------+----------+
|1           |5       |5   |Entrega antes do previsto.     |2024-03-09|
|2           |6       |5   |Boa qualidade pelo preço.      |2023-01-08|
|3           |8       |5   |Excelente produto!             |2022-09-11|
|4           |9       |1   |Veio amassado.                 |2024-02-18|
|5           |10      |1   |Produto diferente da descrição.|2022-05-24|
+------------+--------+----+-------------------------------+----------+
only showing top 5 rows

=== FORNECEDORES ===
root
 |-- fornecedor_id: integer (nullable = true)
 |-- razao_social: string (nullable = true)
 |-- cnpj: string (nullable = true)
 |-- regiao: string (nullable = true)
 |-- cidade: string (nullable = true)
 |-- contato: string (nullable = true)
 |-- segmento: string (nullable = true)

+-------------+----

In [11]:
# Histórico de versões Delta
from delta.tables import DeltaTable
for table in ['clientes', 'produtos', 'vendas', 'fretes', 'pagamentos', 'cupons', 'avaliacoes', 'fornecedores']:
    dt   = DeltaTable.forPath(spark, f'{BRONZE}/{table}')
    hist = dt.history(1).select('version', 'timestamp', 'operation').collect()[0]
    print(f'[{table}] versão={hist["version"]}  op={hist["operation"]}')

[clientes] versão=4  op=WRITE
[produtos] versão=4  op=WRITE
[vendas] versão=4  op=WRITE
[fretes] versão=3  op=WRITE
[pagamentos] versão=3  op=WRITE
[cupons] versão=3  op=WRITE
[avaliacoes] versão=3  op=WRITE
[fornecedores] versão=0  op=WRITE
